# Exercise 2: Deep Learning Classification (EEG → Sleep Staging)

## From Machine Learning to Deep Learning: When and Why?

### The Evolution: Hand-Crafted Features vs Learned Representations

In **Exercise 1**, we tackled sleep staging using **traditional machine learning** pipelines:

```
Raw EEG → Feature Engineering (PSD) → Manual Feature Selection (bands/PCA) → Classifier
```

This approach required:
- **Domain expertise**: Understanding which frequency bands matter (delta, theta, alpha, sigma, beta)
- **Feature engineering**: Computing PSD, averaging bands, dimensionality reduction
- **Pipeline design**: Carefully orchestrating preprocessing, scaling, and classification steps

In **Exercise 2**, we explore **deep learning**, which learns features automatically:

```
Raw EEG Signal → Neural Network → Sleep Stage Prediction
```

The network learns directly from raw time-domain signals, discovering relevant patterns without explicit feature engineering.

---

## When is Deep Learning Worth Considering?

### Deep Learning is Preferable When:

1. **Large datasets are available** (thousands to millions of samples)
2. **Raw signal patterns are complex and hierarchical**
   - EEG contains multi-scale temporal patterns (spindles, K-complexes, slow waves)
   - Convolutional layers naturally capture local patterns → global structures
   - Traditional features may miss subtle discriminative patterns

3. **Feature engineering is difficult or requires extensive domain knowledge**
   - Discovering optimal spectral bands, time windows, or transformations is challenging
   - Networks can learn task-specific representations automatically
   - Reduces dependency on expert knowledge for feature design

4. **Computational resources are available**
   - GPUs/TPUs accelerate training (otherwise prohibitively slow) -> Turn on the GPU RunTime on Colab!
   - Training time: hours to days vs minutes for traditional ML

### Traditional ML (Exercise 1) is Preferable When:

1. **Limited data** (<1000 samples per class)
2. **Interpretability is critical**
4. **Domain knowledge provides strong priors**
   - Well-established features (PSD bands) are known to work
   - No need to rediscover what's already known
   - Can combine domain knowledge with data-driven selection (e.g., PCA)

---

## This Exercise: Deep Learning for Sleep Staging

### What We'll Explore:

- **Architecture**: SimpleEEGNet, a lightweight convolutional neural network designed for EEG
- **Input**: Raw time-domain signals (3000 samples × 1 channel per 30-second epoch)
- **Training strategy**: Leave-one-subject-out cross-validation with early stopping on validation set
- **Comparison**: How does end-to-end learning compare to PSD + PCA + SVM from Exercise 1?


In [59]:
%matplotlib inline
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    get_scorer,
)

from pickle import dump, load

import importlib
import model
importlib.reload(model)

from model import get_simple_eegnet


## Constants and Random Seed Configuration

Define global constants for sleep stage labels, sampling frequency, and set random seeds for reproducibility across PyTorch and NumPy operations.


In [60]:
# Sleep stage labels
STAGES = ["Wake", "N1", "N2", "N3", "REM"]

# Sampling frequency in Hz
fs = 100

# Random seed for reproducibility
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Exercise 2.1: Data Loading and Visualization Functions

Unlike Exercise 1 where we worked with pre-computed PSD features, here we load **raw EEG signals** directly. This allows the neural network to learn temporal patterns end-to-end without manual feature engineering.

### Key Differences from Exercise 1:

| Aspect | Exercise 1 (Traditional ML) | Exercise 2 (Deep Learning) |
|--------|----------------------------|---------------------------|
| **Input data** | PSD features (~500 frequency bins) | Raw signals (3000 time points) |
| **Data type** | Frequency domain (µV²/Hz) | Time domain (µV) |
| **Preprocessing** | Log scaling + standardization | Standardization only |
| **Feature extraction** | Manual (band averaging or PCA) | Automatic (learned by network) |
| **Input shape** | (n_epochs, n_frequencies) | (n_epochs, n_timepoints) |

### Functions Overview:

- **`read_subject_signal()`**: Loads raw EEG time series instead of PSD
- **`plot_signal()`**: Visualizes 30-second epochs in the time domain to inspect signal quality and stage-specific patterns


In [61]:
def read_subject_signal(i: int) -> Tuple[np.array, np.array]:
    """
    Load raw EEG signal and sleep stage labels for a specific subject.
    
    Reads pre-processed single-channel EEG recordings (Fpz-Cz derivation) saved as
    PyTorch tensors. Each epoch represents 30 seconds of continuous recording sampled
    at 100 Hz, resulting in 3000 time points per epoch.

    Parameters
    ----------
    i : int
        Subject index (0-based indexing).

    Returns
    -------
    signal : np.array
        2D array of shape (n_epochs, n_timepoints) containing raw EEG amplitude values
        in microvolts (µV). For standard 30-second epochs at 100 Hz sampling rate,
        n_timepoints = 3000.
    stages : np.array
        1D array of shape (n_epochs,) containing sleep stage labels (0=Wake, 1=N1,
        2=N2, 3=N3, 4=REM). Labels are converted from original 1-5 encoding to 0-4
        for zero-based indexing.

    Notes
    -----
    - Signal derivation: Fpz-Cz (single-channel)
    - Sampling rate: 100 Hz
    - Epoch duration: 30 seconds
    - Signal units: microvolts (µV)
    - Original stage encoding (1-5) is converted to (0-4) for consistency with
      Python's zero-based indexing

    Examples
    --------
    >>> signal, stages = read_subject_signal(0)
    >>> print(f"Subject 0: {signal.shape[0]} epochs of {signal.shape[1]} samples")
    >>> print(f"Sampling rate: {signal.shape[1] / 30} Hz")
    >>> print(f"Stage distribution: {np.bincount(stages)}")
    """

    signal = (
        torch.load(f"data/signals/subject_{i}.pt").float().numpy().reshape(-1, 3000)
    )
    stages = (
        torch.load(f"data/stages/subject_{i}.pt").long().numpy() - 1
    )  # stages are from 1 to 5, we convert them to 0 to 4 for easier indexing

    return signal, stages


def plot_signal(
    signal: np.array,
    stage: int | None = None,
    fs: int = 100,
) -> None:
    """
    Visualize a single 30-second EEG epoch with optional stage annotation.
    
    Creates a time-domain plot of raw EEG signal amplitude with centered baseline,
    grid overlay, and sleep stage label. Useful for visual inspection of signal
    quality and stage-specific EEG characteristics.

    Parameters
    ----------
    signal : np.array
        EEG signal array of shape (n_timepoints,) or (1, n_timepoints) containing
        amplitude values in microvolts (µV). For standard epochs, n_timepoints = 3000.
    stage : int | None, optional
        Sleep stage label (0-4 corresponding to Wake, N1, N2, N3, REM). If None,
        the plot title will not include stage information. Default is None.
    fs : int, optional
        Sampling frequency in Hz. Used to convert sample indices to time in seconds.
        Default is 100 Hz.

    Returns
    -------
    None
        Displays the plot using matplotlib.pyplot.show().

    Notes
    -----
    - The y-axis baseline is centered at zero amplitude for easier visual interpretation
    - Time axis is displayed in seconds (0-30 for standard epochs)
    - Grid overlay aids in identifying specific time points and amplitude ranges

    Examples
    --------
    >>> signal, stages = read_subject_signal(0)
    >>> plot_signal(signal[0], stage=stages[0], fs=100)  # Plot first epoch
    >>> plot_signal(signal[100])  # Plot without stage label
    """

    pass


## Exercise 2.2: Load Training Data and Visualize Sample Epoch

Load raw EEG signals for the first 10 subjects (training pool) and visualize a representative epoch to understand the input data characteristics.

### What to Look For:

- **Amplitude range**: Typical EEG ranges from -100 to +100 µV
- **Frequency content**: Visual inspection can reveal dominant rhythms (e.g., alpha waves in Wake appear as regular ~10 Hz oscillations)
- **Artifacts**: Check for extreme values, flatlines, or non-physiological patterns
- **Stage-specific patterns**: 
  - **Wake**: Mixed frequencies, possible eye movement artifacts
  - **N1**: Theta activity (4-8 Hz), low amplitude
  - **N2**: Sleep spindles (burst of ~12-14 Hz), K-complexes (sharp waves)
  - **N3**: High-amplitude slow waves (delta <4 Hz)
  - **REM**: Low amplitude, mixed frequencies similar to Wake but with muscle atonia

This visual inspection helps validate data quality before training.


In [62]:
# read the data of the first 10 subjects X : training data, y : labels 

X, y = [], []

# plot epochs from the subjects to check any clear artifact mirroring a sleep stage
# plot_signal(epoch_to_plot, stage_to_plot, fs)

## Exercise 2.3: Create Validation Set for Checkpointing

### The Neural Network Training Challenge: When to Stop?

Unlike traditional ML classifiers (which have closed-form solutions or clear convergence criteria), neural networks train iteratively through gradient descent. This creates a critical problem:

**Overfitting during training**: As training progresses, the model improves on the training set but may start memorizing noise, degrading performance on unseen data.

### The Solution: Validation-Based Checkpointing

We split our 10 subjects into:
- **Training subjects (8)**: Used for gradient descent updates
- **Validation subjects (2)**: Monitor generalization performance

**Early stopping strategy**:
1. After each training epoch, evaluate loss/accuracy on the validation set
2. Save the model when validation performance improves
4. Restore the best model (based on validation performance)

### Why Not Use Test Subjects (11-15) for Validation?

**Critical principle**: Test subjects must remain completely unseen until final evaluation. Using them for early stopping would:
- Leak information into model selection
- Overestimate true generalization performance
- Violate the independence of the test set

### Data Split Strategy:

```
Subjects 0-9:  Training pool (10 subjects)
  ├─ Train:    8 subjects (for gradient updates)
  └─ Val:      2 subjects (for early stopping)

Subjects 10-14: Test set (5 subjects, completely held out)
```

This three-way split ensures honest evaluation while preventing overfitting during training.


In [63]:
# To train a neural network model we need to have a stopping criterion to avoid overfitting over the training epochs.
# We will use a validation set for this purpose.

# Define validation set by randomly selecting 2 subjects from the training pool
# you can use np.random.choice to select 2 random subjects

train_subjects = []
val_subjects = []

## Exercise 2.4: Leave-One-Subject-Out Cross-Validation for Neural Networks

### Adapting Cross-Validation to Deep Learning

In Exercise 1, we used scikit-learn's `cross_validate()` function which handled all the complexity. For deep learning, we must implement the cross-validation loop manually because the `skorch` library that we use for training the model doesn't directly support the group-cross-validation functionalities of sklearn.

So we need to manually implement the `cross_validate()` logic:

For each training subject `i`:
```
Fold i:
  ├─ Test:       Subject i (held out for this fold)
  ├─ Train:      All other training subjects except i and validation subjects
  └─ Validation: 2 randomly selected subjects
```

Note: Validation subjects are the same across all folds (selected once at the beginning).

### Hyperparameters to Tune:

- **`batch_size`**: Number of epochs processed together (affects memory and convergence)
  - Smaller: More noise in gradients, better generalization, slower training
  - Larger: Smoother gradients, faster training, risk of overfitting
  - Typical: 16-128 for EEG

- **`max_epochs`**: Maximum training iterations over the full training set
  - With early stopping, actual training may stop earlier
  - Typical: 20-100 for small datasets

- **`learning_rate`**: Step size for gradient descent
  - Too high: Unstable training, divergence
  - Too low: Slow convergence, may get stuck
  - Typical: 1e-4 to 1e-2 (often start with 1e-3)

### Preprocessing: Standardization

We apply standardization in the **time domain** (per time point across epochs), not in the frequency domain.

**Why standardize?**
- Neural networks are sensitive to input scale
- Helps gradient descent converge faster
- Different subjects may have different baseline amplitudes

**Pipeline**: `raw_signal → standardize (fit on train, transform on val/test) → network`

In [64]:
# In general there are many other parameters to tune such as the learning_rate (lr), batch_size, max_epochs etc...
# We can decide those parameters using a cross-validation approach as we did in the previous exercise

# Define hyperparameters for neural network training
batch_size = 32
max_epochs = 20
learning_rate = 1e-3

scoring_metrics = [
    "accuracy",
    "f1_weighted",
    "balanced_accuracy",
    "precision_weighted",
    "recall_weighted",
]

def cross_validate_eegnet(
    X : list[np.array], 
    y : list[np.array],
    train_subjects : list[int] = train_subjects,
    val_subjects : list[int] = val_subjects,
    max_epochs : int = max_epochs,
    batch_size : int = batch_size,
    learning_rate : float = learning_rate,
    scoring_metrics : list[str] = scoring_metrics,
    verbose : int = 1
    ):
    """
    Perform leave-one-subject-out cross-validation for neural network sleep staging.
    
    Implements manual cross-validation for deep learning models using skorch, which
    does not natively support grouped cross-validation. Each fold trains on n-1
    training subjects, validates on held-out validation subjects for early stopping,
    and tests on the remaining training subject.

    Parameters
    ----------
    X : list[np.array]
        List of feature arrays, one per subject. Each array has shape (n_epochs, n_timepoints)
        containing raw EEG signal data in microvolts.
    y : list[np.array]
        List of label arrays, one per subject. Each array has shape (n_epochs,) with
        sleep stage labels (0=Wake, 1=N1, 2=N2, 3=N3, 4=REM).
    train_subjects : list[int], optional
        Indices of subjects to use in the cross-validation loop. Each will serve as
        the test subject for one fold. Default uses the global `train_subjects` variable.
    val_subjects : list[int], optional
        Indices of subjects reserved for validation/early stopping. These subjects are
        never used for testing and remain constant across all folds. Default uses the
        global `val_subjects` variable.
    max_epochs : int, optional
        Maximum number of training epochs per fold. Actual training may stop earlier
        due to early stopping based on validation performance. Default is 20.
    batch_size : int, optional
        Number of samples per gradient update. Affects training speed and convergence.
        Default is 32.
    learning_rate : float, optional
        Step size for gradient descent optimization. Default is 1e-3.
    scoring_metrics : list[str], optional
        List of sklearn-compatible metric names to compute. Default includes
        'accuracy', 'f1_weighted', 'balanced_accuracy', 'precision_weighted',
        and 'recall_weighted'.
    verbose : int, optional
        Verbosity level for training output. 0 = silent, 1 = progress bar, 
        2+ = detailed logs. Default is 1.

    Returns
    -------
    results : list[dict]
        List of dictionaries, one per fold, containing:
        - 'test_subject': Subject index used for testing in this fold
        - 'test_{metric}': Performance on test subject for each metric
        - 'val_{metric}': Performance on validation subjects for each metric
    y_true_all : list
        Concatenated true labels from all test folds (n_total_test_epochs,).
    y_pred_all : list
        Concatenated predictions from all test folds (n_total_test_epochs,).

    Notes
    -----
    **Cross-Validation Strategy:**
    - For each subject i in train_subjects:
      - Test set: Subject i
      - Training set: All subjects in train_subjects except i and val_subjects
      - Validation set: val_subjects (constant across folds)
    
    **Preprocessing Pipeline:**
    - StandardScaler fitted on training data only
    - Applied to validation and test data using training statistics
    - Prevents data leakage while normalizing signals
    
    **Model Training:**
    - Uses get_simple_eegnet() to create a fresh model for each fold
    - Early stopping monitors validation loss
    - Best model (by validation performance) is used for test predictions

    **Metric Computation:**
    - Uses sklearn's get_scorer() to obtain callable scorers
    - Metrics computed on both validation and test sets
    - Results stored per-fold for analysis

    Examples
    --------
    >>> # Perform cross-validation with default parameters
    >>> results, y_true, y_pred = cross_validate_eegnet(X, y)
    >>> 
    >>> # With custom hyperparameters
    >>> results, y_true, y_pred = cross_validate_eegnet(
    ...     X, y, 
    ...     max_epochs=50, 
    ...     batch_size=64, 
    ...     learning_rate=5e-4,
    ...     verbose=0
    ... )
    >>> 
    >>> # Analyze results
    >>> results_df = pd.DataFrame(results)
    >>> print(f"Mean test accuracy: {results_df['test_accuracy'].mean():.3f}")
    """

    results = []
    y_true_all = []
    y_pred_all = []

    for i in tqdm( train_subjects, total=len(train_subjects), desc="LOSO Cross Validation"):
        X_train, X_val, X_test = np.empty()
        y_train, y_val, y_test = np.empty()
        
        # The i-th subject is taken out as testing subject
        
        # The rest of the subjects are used for training

        # And we need also the validation subjects

        # Concatenate the data
        # Now we need to scale them properly using the standard scaler
        # note: u need to reshape to 2D for the scaler and then reshape back to its original shape

        # Now we can create the model and train it with
        # get_simple_eegnet(...) function
        # call the .fit method with X_train, y_train to train it

        # compute predictions on the test subject
        
        # compute the scoring metrics
        # use get_scorer from sklearn to get the scoring functions
        # then apply them using scorer(model, X_test, y_test)

        
    return results, y_true_all, y_pred_all

# suggeted to run with verbose = 0 to avoid too much output
results, y_true_all, y_pred_all = cross_validate_eegnet(X, y, verbose = 0)

LOSO Cross Validation: 0it [00:00, ?it/s]


In [65]:
# visualize results in a dataframe

## Exercise 2.5: Evaluate Cross-Validation Results

### Interpreting the Confusion Matrix

The confusion matrix reveals which sleep stages are most challenging for the neural network and common misclassification patterns.

**What to look for**:

1. **Diagonal strength**: High values indicate accurate stage identification
2. **Adjacent stage confusion**: N1 ↔ N2 confusion is expected (transitional stages with similar EEG)
3. **Wake vs REM confusion**: Both have desynchronized EEG with mixed frequencies
4. **N3 performance**: Should be easiest to detect (distinctive high-amplitude slow waves)

### Classification Report Metrics

**Per-class metrics**:
- **Precision**: Of all predicted N2 epochs, what fraction were actually N2?
  - High precision: Few false alarms for this stage
  - Low precision: Over-predicting this stage
  
- **Recall**: Of all actual N2 epochs, what fraction did we detect?
  - High recall: Not missing many instances of this stage
  - Low recall: Under-detecting this stage

- **F1-score**: Harmonic mean balancing precision and recall
  - Best metric when classes are imbalanced

**Aggregate metrics**:
- **Macro average**: Simple average across all stages (treats each stage equally)
- **Weighted average**: Accounts for class imbalance (more realistic for deployment)

### Comparison to Exercise 1

After reviewing the results, consider:
- **Accuracy**: How does SimpleEEGNet compare to the best traditional ML pipeline (SVM + PCA)?
- **Per-stage performance**: Are there stages where deep learning excels or struggles?
- **Consistency**: Is variance across folds higher or lower than traditional ML?
- **Training effort**: Was the extra complexity of neural network training worth the performance gain?


In [66]:
# Visualize cross-validation results with confusion matrix and classification report

# plot the confusion matrix with ConfusionMatrixDisplay.from_predictions 


# plot the classification report as a dataframe ( output_dict=True in classification_report )

# convert the classification report dictionary to a pandas dataframe for better visualization 
# you can transpose it to have the classes as rows report_df.T


## Exercise 2.6: Train Final Model and Persist to Disk

### From Cross-Validation to Deployment

Cross-validation provided an **estimate** of generalization performance. Now we train the **final production model** using all available training data.

### Training Strategy:

**Training data**: All 8 training subjects (excluding the 2 validation subjects)
- Use the same subjects that were in the training folds during CV
- This maximizes the amount of data for learning robust representations

**Validation data**: The same 2 subjects held out during CV
- Used for early stopping to prevent overfitting
- Ensures consistent hyperparameter selection

**Hyperparameters**: Use the values tuned during cross-validation exploration
- `batch_size = 32`
- `max_epochs = 20`
- `learning_rate = 1e-3`

### Why Report Training and Validation Accuracy?

1. **Training accuracy**: Shows model capacity
   - Should be high (>80%) if network is expressive enough
   - Too low: Model is underfitting (increase capacity or train longer)
   - ~100%: Potential overfitting (check validation accuracy)

2. **Validation accuracy**: Estimates generalization
   - Should be close to training accuracy (within 5-10%)
   - Large gap: Overfitting (reduce capacity, add regularization, or get more data)
   - Similar to training: Good generalization

### Model Persistence with Pickle

Serializing the trained model enables:
- **Reproducibility**: Same predictions without retraining
- **Deployment**: Load in production environments (servers, edge devices)
- **Sharing**: Distribute models to collaborators
- **Versioning**: Compare different model versions

**Important**: The pickle file contains:
- Network architecture (layer types, sizes)
- Trained weights (learned parameters)
- Optimizer state (not needed for inference)
- Any preprocessing fitted on training data (e.g., scaler parameters)

**Note**: Pickle files are Python-specific. For cross-platform deployment, consider:
- ONNX (Open Neural Network Exchange)
- TorchScript (PyTorch's serialization format)
- Explicit weight export (e.g., HDF5, NumPy arrays)


In [67]:
# Once we are happy with the model and we chose the best parameters we need to deploy it

# We retrain the model on all the data using the best parameters found
batch_size = 32
max_epochs = 20
learning_rate = 1e-3

# Define the training data and fit the scaler
X_train, y_train = [], []

# Define the validation data and scale it properly
X_val, y_val = [], []

# build the neural network
# use get_simple_eegnet function

# train the model on all the training data

# Evaluate the training accuracy

# Save the model with pickle
# with open("best_simple_eegnet.pkl", "wb") as f:
#    dump(best_net, f)


## Exercise 2.7: Evaluate on Completely Unseen Test Subjects

### The Final Test: True Generalization Performance

Subjects 10-14 have been completely held out throughout the entire process:
- Not used in any training fold
- Not used for validation or early stopping
- Not used for hyperparameter selection

This represents the **most realistic evaluation** of how the model will perform in deployment on genuinely new patients.

### What to Evaluate:

After loading the test data, we should:

1. **Apply the same preprocessing**: Use the scaler fitted on training data
   ```python
   X_test_scaled = scaler.transform(X_test)  # NEVER fit on test data!
   ```

2. **Generate predictions**: Run the model on scaled test data
   ```python
   y_pred = best_net.predict(X_test_scaled)
   ```

3. **Compute metrics**:
   - Confusion matrix (normalized by true labels)
   - Classification report (precision, recall, F1 per stage)
   - Overall accuracy

4. **Compare to cross-validation estimates**:
   - Test accuracy should be close to CV accuracy (within a few %)
   - Large discrepancy suggests:
     - Validation subjects were not representative
     - Overfitting to the training pool
     - Hyperparameter selection bias

### Critical Reminder: Never Fit on Test Data

Common mistakes to avoid:
- ❌ `scaler.fit_transform(X_test)` → Leaks test information!
- ❌ Adjusting hyperparameters based on test performance → Indirect overfitting!
- ✅ `scaler.transform(X_test)` → Uses training statistics only
- ✅ Report test results exactly once → Prevents selection bias

### Expected Next Steps:

After evaluating on test subjects, the notebook should:
- Display confusion matrix for test set
- Print classification report
- Compare test accuracy to CV accuracy
- Discuss whether deep learning outperformed traditional ML (Exercise 1)


In [68]:
# Load the model and test on the testing subjects (11-15)
# with open("best_simple_eegnet.pkl", "rb") as f:
#    best_net = load(f)


X_test, y_test = [], []
y_pred_all = []
y_true_all = []


# plot confusion matrix

# print classification report
